# 데이터 개요

2023 스마트농업 AI 경진대회 데이터셋의 구조와 타겟 분포를 살펴봅니다. 이 대회는 농가 운영 데이터로부터 두 개의 누적 타겟 — 수확량(`outtrn_cumsum`) 과 난방 에너지 사용량(`HeatingEnergyUsage_cumsum`) — 을 동시에 예측해야 하는 multi-target 회귀 문제입니다.


## Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)

## 데이터 로드

대회 데이터(`2023_smartFarm_AI_hackathon_dataset.csv`)는 `data/` 경로에 둡니다.


In [ ]:
DATA_DIR = Path("data")
INPUT_CSV = DATA_DIR / "2023_smartFarm_AI_hackathon_dataset.csv"

raw = pd.read_csv(INPUT_CSV)
print(f"shape: {raw.shape}")
raw.head()

## 컬럼 정보 요약

In [ ]:
dtype_summary = (
    raw.dtypes.rename("dtype")
    .to_frame()
    .assign(non_null=raw.notna().sum(), unique=raw.nunique())
    .sort_values("unique")
)
dtype_summary

## 단일값(상수) 컬럼 식별

unique 값이 1개인 컬럼은 학습에 기여하지 않으므로 후속 단계에서 제거합니다.


In [ ]:
constant_columns = dtype_summary[dtype_summary["unique"] <= 1].index.tolist()
print(f"constant columns ({len(constant_columns)}):", constant_columns)

## 타겟 분포

In [ ]:
TARGETS = ["outtrn_cumsum", "HeatingEnergyUsage_cumsum"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, target in zip(axes, TARGETS):
    sns.histplot(raw[target], kde=True, ax=ax, color="steelblue")
    ax.set_title(f"{target} 분포")
plt.tight_layout()
plt.show()

raw[TARGETS].describe()

## 농가별 타겟 추이

`frmDist` (농가 식별자) 가 있다면 농가별 누적 타겟 추이를 살펴봅니다.


In [ ]:
if "frmDist" in raw.columns and "date" in raw.columns:
    sample_farms = raw["frmDist"].unique()[:5]
    sample = raw[raw["frmDist"].isin(sample_farms)].copy()
    sample["date"] = pd.to_datetime(sample["date"], format="%Y%m%d", errors="coerce")

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    for target, ax in zip(TARGETS, axes):
        for farm_id, group in sample.groupby("frmDist"):
            ax.plot(group["date"], group[target], label=f"farm {farm_id}", alpha=0.7)
        ax.set_ylabel(target)
        ax.legend(loc="upper left", fontsize=8)
    plt.suptitle("농가별 누적 타겟 추이 (sample)")
    plt.tight_layout()
    plt.show()

## 결측치 점검

In [ ]:
missing = raw.isna().sum()
missing[missing > 0].sort_values(ascending=False)

## 정리

- 행 수와 컬럼 수, 각 타겟의 분포 형태와 자릿수를 확인했습니다.
- unique 값이 1개인 상수 컬럼은 후속 노트북에서 제거합니다.
- 두 타겟의 스케일이 매우 다르므로 모델 비교 시 RMSE 외에 R² 도 함께 사용합니다.
